# PINNProof package smoke tests

This notebook is designed as an **examples-level regression smoke test** for the reorganized `pinnproof` package.

It checks:
- Public API imports
- Validation metrics outputs (`rmse`, `mae`, `nrmse`, `trajectory_metrics`)
- Verification utilities (`finite_difference`, `swing_equation_residual`, `verify_swing_trajectories`)
- Pass/fail behavior on both a physically-consistent and intentionally-inconsistent trajectory

Use this notebook as a baseline demo for new users and as a quick branch sanity check after refactors.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import numpy as np

for base in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (base / 'examples' / 'path_setup.py').is_file():
        repo_root = base
        if str(repo_root) not in sys.path:
            sys.path.insert(0, str(repo_root))
        break
else:
    raise ModuleNotFoundError('Could not locate the PINNProof repository root relative to the notebook working directory.')

from examples.path_setup import configure_notebook_paths

configure_notebook_paths()

from pinnproof import (
    VerificationReport,
    mae,
    nrmse,
    rmse,
    trajectory_metrics,
    verify_swing_trajectories,
)
from pinnproof.verification.residuals import finite_difference, swing_equation_residual

print('Imports succeeded ✅')

## 1) Public API smoke check

In [ ]:
expected_public = {
    'rmse',
    'mae',
    'nrmse',
    'trajectory_metrics',
    'VerificationReport',
    'verify_swing_trajectories',
}

actual_public = set(__import__('pinnproof').__all__)
missing = expected_public - actual_public
assert not missing, f'Missing public exports: {missing}'
print('Public API exports look good ✅')

## 2) Validation metrics checks

In [ ]:
rng = np.random.default_rng(7)
y_true = rng.normal(size=(4, 15, 3))
y_pred = y_true + 0.05 * rng.normal(size=(4, 15, 3))

rmse_val = rmse(y_true, y_pred)
mae_val = mae(y_true, y_pred)
nrmse_val = nrmse(y_true, y_pred)
metrics = trajectory_metrics(y_true, y_pred)

manual_rmse = np.sqrt(np.mean((y_pred - y_true) ** 2))
manual_mae = np.mean(np.abs(y_pred - y_true))
manual_nrmse = manual_rmse / (np.ptp(y_true) + 1e-12)

assert np.isclose(rmse_val, manual_rmse)
assert np.isclose(mae_val, manual_mae)
assert np.isclose(nrmse_val, manual_nrmse)
assert set(metrics) == {
    'global_rmse',
    'global_mae',
    'per_state_rmse',
    'per_state_mae',
    'per_trajectory_rmse',
}
assert metrics['per_state_rmse'].shape == (3,)
assert metrics['per_state_mae'].shape == (3,)
assert metrics['per_trajectory_rmse'].shape == (4,)

print('Metric values and shapes verified ✅')
print({k: (v if np.isscalar(v) else v.shape) for k, v in metrics.items()})

In [ ]:
# Negative test: shape mismatch should raise ValueError
raised = False
try:
    trajectory_metrics(y_true, y_pred[:, :, :2])
except ValueError:
    raised = True

assert raised, 'trajectory_metrics should reject mismatched shapes'
print('Shape mismatch error behavior verified ✅')

## 3) Verification residual checks

In [ ]:
times = np.linspace(0.0, 1.0, 501)
omega = np.sin(times)
domega_dt = np.cos(times)

inertia = 0.4
damping = 0.1
coupling = 2.0
mechanical_power = 0.3

# Build delta so that residual is (approximately) zero by construction
rhs = mechanical_power - inertia * domega_dt - damping * omega
assert np.max(np.abs(rhs / coupling)) <= 1.0, 'arcsin domain violation: adjust parameters'
delta = np.arcsin(rhs / coupling)

residual = swing_equation_residual(
    delta=delta,
    omega=omega,
    times=times,
    inertia=inertia,
    damping=damping,
    coupling=coupling,
    mechanical_power=mechanical_power,
)

assert residual.shape == omega.shape
assert float(np.max(np.abs(residual))) < 2e-3

fd = finite_difference(omega, times)
assert np.max(np.abs(fd - domega_dt)) < 1e-2

print('Residual and finite-difference behavior verified ✅')

In [ ]:
report_pass = verify_swing_trajectories(
    delta=delta,
    omega=omega,
    times=times,
    inertia=inertia,
    damping=damping,
    coupling=coupling,
    mechanical_power=mechanical_power,
    tolerance=5e-3,
)

assert isinstance(report_pass, VerificationReport)
assert report_pass.passed is True

# Perturb delta to intentionally break physical consistency
bad_delta = delta + 0.3
report_fail = verify_swing_trajectories(
    delta=bad_delta,
    omega=omega,
    times=times,
    inertia=inertia,
    damping=damping,
    coupling=coupling,
    mechanical_power=mechanical_power,
    tolerance=5e-3,
)
assert report_fail.passed is False

print('Pass/fail verification report behavior looks good ✅')
print('PASS report:', report_pass)
print('FAIL report:', report_fail)

## 4) Recommended usage pattern in CI / local checks

- Keep this notebook in `examples/` for user education.
- Mirror these assertions in small `pytest` tests when you are ready to add CI.
- Run this notebook after refactors affecting package API or numerical utilities.